## Setting up Laya

In [5]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [6]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Please switch your Kaggle Accelerator: on the right sidebar, go to Notebook options -> "
    "Accelerator -> select GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")

Sun Sep 27 13:17:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!pip install laya

In [8]:
import laya
from laya import Router

# Preload checkpoints into memory for instant sub-35ms routing
router = Router(preload=True)

state = {
    "from": "user@acme.com",
    "subject": "Duplicate charge on invoice #4411",
    "body": "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
}

questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "sales": "pricing, new contracts",
            "other": "everything else"
        }
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["not urgent", "soon", "critical deadline or blocking issue"]
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the user threaten to cancel or leave?"
    },
    "refund_requested": {
        "type": "noul",
        "instructions": "Does the user explicitly request a refund?"
    }
}

# 1. English state -> automatically routed to ModernBERT-large (39.5 ms)
res_en = router.predict(state, questions)
print("Department :", res_en["answers"]["department"]["choice"])  # -> billing (confidence: 0.94)
print("Routing    :", res_en["routing"]["model"])                 # -> english

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/laya/router.py:260: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, device=self.device, token=self.token, subfolder=sub)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Department : billing
Routing    : english


## Load Datasets

In [9]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [10]:
import pandas as pd

BASE_PATH = "/kaggle/input/datasets/priyamsaha17/generalmind-27-sept-datasets/datasets/rule_engine"

# Load JSONL files
finetune_df = pd.read_json(f"{BASE_PATH}/fine_tuning.jsonl", lines=True)
validation_df = pd.read_json(f"{BASE_PATH}/validation.jsonl", lines=True)

print("Fine-tuning shape:", finetune_df.shape)
print("Validation shape:", validation_df.shape)

Fine-tuning shape: (200, 4)
Validation shape: (50, 3)


In [11]:
finetune_df.head(1)

case_id  \
0  case_a7ae6c5453b44045   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [12]:
validation_df.head(1)

case_id  \
0  case_ab6e3f1b9c2e7640   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

## Zero Shot Attempt

In [13]:
# question 


questions = {
    "R01_AMENDMENT": {
        "type": "choice",
        "instructions": "Should the requested amendment be applied to the order?",
        "criteria": {
            "APPLY": "Apply the requested amendment.",
            "DO_NOT_APPLY": "Do not apply the requested amendment.",
            "REVIEW": "Insufficient certainty; send to a human for review.",
            "NOT_APPLICABLE": "This case does not involve an amendment."
        }
    },

    "R02_SUBSTITUTION": {
        "type": "choice",
        "instructions": "Should the requested production substitution be accepted?",
        "criteria": {
            "ACCEPT": "Accept the requested production substitute.",
            "REJECT": "Reject the requested substitute.",
            "REVIEW": "Needs human review.",
            "NOT_APPLICABLE": "No substitution decision is needed."
        }
    },

    "R03_RELEASE": {
        "type": "choice",
        "instructions": "Should the order be released for dispatch or held?",
        "criteria": {
            "RELEASE": "Release the order for dispatch.",
            "HOLD": "Hold the order instead of releasing it.",
            "REVIEW": "Needs human review before dispatch.",
            "NOT_APPLICABLE": "Dispatch is not relevant."
        }
    },

    "R04_DOCUMENT_RELATION": {
        "type": "choice",
        "instructions": "What is the relationship between this document and previous billing obligations?",
        "criteria": {
            "DUPLICATE": "This document duplicates a previous billing obligation.",
            "CORRECTION": "This document corrects a previous document.",
            "SEPARATE_TRANSACTION": "This represents a new and separate transaction.",
            "REVIEW": "The relationship is uncertain."
        }
    },

    "R05_GOVERNING_TERMS": {
        "type": "choice",
        "instructions": "Which commercial terms should govern this case?",
        "criteria": {
            "ORIGINAL": "The original commercial terms apply.",
            "REVISED": "The revised commercial terms apply.",
            "MIXED": "The governing terms come from multiple documents.",
            "REVIEW": "The governing terms require human review."
        }
    }
}

In [14]:
# Trying first row of validation

sample = validation_df.iloc[0]

In [15]:
state = {
    "document_context": sample["input"]["document_context"],
    "email_thread": sample["input"]["email_thread"],
    "document_text": sample["input"]["document_text"],
    "reference_records": sample["input"]["reference_records"],
    "decision_policy": sample["input"]["decision_policy"]
}

#state

In [16]:
prediction_default = router.predict(state, questions, max_len=512)

In [17]:
prediction_default["answers"]["R01_AMENDMENT"]["choice"]

'APPLY'

In [18]:
print(f"Case ID: {sample['case_id']}")

print("\nGround Truth")
ground_truth = sample["target"]["rule_outputs"]

for k, v in ground_truth.items():
    print(f"{k:25} {v}")

print("\nLaya Zero-Shot Prediction (Default)")

# If using router.predict() from the current Laya version,
# predictions are under prediction["answers"]
answers = prediction_default["answers"] if "answers" in prediction_default else prediction_default

for name in questions.keys():
    ans = answers[name]
    print(
        f"{name:25} "
        f"{ans['choice']:20} "
        f"(confidence={ans['confidence']:.3f})"
    )

Case ID: case_ab6e3f1b9c2e7640

Ground Truth
R01_AMENDMENT             DO_NOT_APPLY
R02_SUBSTITUTION          REJECT
R03_RELEASE               NOT_APPLICABLE
R04_DOCUMENT_RELATION     CORRECTION
R05_GOVERNING_TERMS       REVIEW

Laya Zero-Shot Prediction (Default)
R01_AMENDMENT             APPLY                (confidence=0.015)
R02_SUBSTITUTION          ACCEPT               (confidence=0.021)
R03_RELEASE               RELEASE              (confidence=0.028)
R04_DOCUMENT_RELATION     CORRECTION           (confidence=0.059)
R05_GOVERNING_TERMS       ORIGINAL             (confidence=0.052)


In [19]:
prediction_td = router.predict(state, questions, max_len=512, model="typed-decisions")

print(f"Case ID: {sample['case_id']}")

print("\nGround Truth")
ground_truth = sample["target"]["rule_outputs"]

for k, v in ground_truth.items():
    print(f"{k:25} {v}")

print("\nLaya Zero-Shot Prediction (Typed Decisions)")

# If using router.predict() from the current Laya version,
# predictions are under prediction["answers"]
answers = prediction_td["answers"] if "answers" in prediction_td else prediction_td

for name in questions.keys():
    ans = answers[name]
    print(
        f"{name:25} "
        f"{ans['choice']:20} "
        f"(confidence={ans['confidence']:.3f})"
    )

Case ID: case_ab6e3f1b9c2e7640

Ground Truth
R01_AMENDMENT             DO_NOT_APPLY
R02_SUBSTITUTION          REJECT
R03_RELEASE               NOT_APPLICABLE
R04_DOCUMENT_RELATION     CORRECTION
R05_GOVERNING_TERMS       REVIEW

Laya Zero-Shot Prediction (Typed Decisions)
R01_AMENDMENT             DO_NOT_APPLY         (confidence=0.014)
R02_SUBSTITUTION          REVIEW               (confidence=0.018)
R03_RELEASE               REVIEW               (confidence=0.025)
R04_DOCUMENT_RELATION     SEPARATE_TRANSACTION (confidence=0.007)
R05_GOVERNING_TERMS       REVIEW               (confidence=0.011)


## Zero Shot Baseline Scores on Validation Set

In [20]:
import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------
# Evaluate one model
# ------------------------------------
def evaluate_model(model_name=None):
    results = []

    for _, row in tqdm(validation_df.iterrows(),
                       total=len(validation_df),
                       desc=model_name or "default"):

        state = {
            "document_context": row["input"]["document_context"],
            "email_thread": row["input"]["email_thread"],
            "document_text": row["input"]["document_text"],
            "reference_records": row["input"]["reference_records"],
            "decision_policy": row["input"]["decision_policy"]
        }

        kwargs = {"max_len": 512}
        if model_name:
            kwargs["model"] = model_name

        pred = router.predict(state, questions, **kwargs)
        answers = pred["answers"]

        gt = row["target"]["rule_outputs"]

        for rule in questions.keys():
            results.append({
                "case_id": row["case_id"],
                "rule": rule,
                "ground_truth": gt[rule],
                "prediction": answers[rule]["choice"],
                "confidence": answers[rule]["confidence"],
                "correct": answers[rule]["choice"] == gt[rule],
                "model": model_name or "default"
            })

    return pd.DataFrame(results)

# ------------------------------------
# Run both models
# ------------------------------------
default_results = evaluate_model()
typed_results   = evaluate_model("typed-decisions")

# ------------------------------------
# Overall accuracy
# ------------------------------------
overall = pd.DataFrame({
    "Model": ["Default", "Typed Decisions"],
    "Accuracy": [
        default_results["correct"].mean(),
        typed_results["correct"].mean()
    ]
})

# ------------------------------------
# Per-rule accuracy
# ------------------------------------
default_rule = (
    default_results.groupby("rule")["correct"]
    .mean()
    .rename("Default")
)

typed_rule = (
    typed_results.groupby("rule")["correct"]
    .mean()
    .rename("Typed Decisions")
)

per_rule = pd.concat([default_rule, typed_rule], axis=1)

# ------------------------------------
# Display
# ------------------------------------
print("\n========== Overall Accuracy ==========\n")
display(overall.style.format({"Accuracy": "{:.2%}"}))

print("\n========== Per-Rule Accuracy ==========\n")
display(per_rule.style.format("{:.2%}"))

# ------------------------------------
# Save predictions
# ------------------------------------
default_results.to_csv("default_zero_shot_predictions.csv", index=False)
typed_results.to_csv("typed_decisions_zero_shot_predictions.csv", index=False)

print("\nSaved:")
print("- default_zero_shot_predictions.csv")
print("- typed_decisions_zero_shot_predictions.csv")

default:   0%|          | 0/50 [00:00<?, ?it/s]

typed-decisions:   0%|          | 0/50 [00:00<?, ?it/s]


========== Overall Accuracy ==========



,Model,Accuracy
0,Default,24.00%
1,Typed Decisions,35.60%



========== Per-Rule Accuracy ==========



,Default,Typed Decisions
rule,,
R01_AMENDMENT,26.00%,44.00%
R02_SUBSTITUTION,30.00%,26.00%
R03_RELEASE,16.00%,22.00%
R04_DOCUMENT_RELATION,26.00%,38.00%
R05_GOVERNING_TERMS,22.00%,48.00%



Saved:
- default_zero_shot_predictions.csv
- typed_decisions_zero_shot_predictions.csv


In [21]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

def evaluate(name, df):
    print("=" * 70)
    print(name)
    print("=" * 70)

    # Overall accuracy
    overall_acc = df["correct"].mean()
    print(f"Overall Accuracy: {overall_acc:.2%}")

    # Per-rule accuracy
    print("\nPer-rule Accuracy")
    rule_acc = df.groupby("rule")["correct"].mean().sort_index()
    print((rule_acc * 100).round(2).astype(str) + "%")

    # Per-rule AUROC
    print("\nPer-rule AUROC")
    aucs = {}

    for rule, g in df.groupby("rule"):
        y_true = g["correct"].astype(int)
        if len(np.unique(y_true)) < 2:
            aucs[rule] = np.nan
        else:
            aucs[rule] = roc_auc_score(y_true, g["confidence"])

    auc_series = pd.Series(aucs).sort_index()
    print(auc_series.round(3))
    print(f"\nMacro AUROC: {auc_series.mean():.3f}")

    return {
        "accuracy": overall_acc,
        "macro_auroc": auc_series.mean(),
        "per_rule_accuracy": rule_acc,
        "per_rule_auroc": auc_series
    }

# Evaluate both models
default_metrics = evaluate("Default", default_results)
typed_metrics   = evaluate("Typed Decisions", typed_results)

# Side-by-side summary
summary = pd.DataFrame({
    "Model": ["Default", "Typed Decisions"],
    "Accuracy": [
        default_metrics["accuracy"],
        typed_metrics["accuracy"]
    ],
    "Macro AUROC": [
        default_metrics["macro_auroc"],
        typed_metrics["macro_auroc"]
    ]
})

print("\n" + "=" * 70)
print("Summary")
print("=" * 70)
display(summary.style.format({
    "Accuracy": "{:.2%}",
    "Macro AUROC": "{:.3f}"
}))

Default
Overall Accuracy: 24.00%

Per-rule Accuracy
rule
R01_AMENDMENT            26.0%
R02_SUBSTITUTION         30.0%
R03_RELEASE              16.0%
R04_DOCUMENT_RELATION    26.0%
R05_GOVERNING_TERMS      22.0%
Name: correct, dtype: object

Per-rule AUROC
R01_AMENDMENT            0.576
R02_SUBSTITUTION         0.442
R03_RELEASE              0.345
R04_DOCUMENT_RELATION    0.605
R05_GOVERNING_TERMS      0.583
dtype: float64

Macro AUROC: 0.510
Typed Decisions
Overall Accuracy: 35.60%

Per-rule Accuracy
rule
R01_AMENDMENT            44.0%
R02_SUBSTITUTION         26.0%
R03_RELEASE              22.0%
R04_DOCUMENT_RELATION    38.0%
R05_GOVERNING_TERMS      48.0%
Name: correct, dtype: object

Per-rule AUROC
R01_AMENDMENT            0.562
R02_SUBSTITUTION         0.552
R03_RELEASE              0.366
R04_DOCUMENT_RELATION    0.610
R05_GOVERNING_TERMS      0.499
dtype: float64

Macro AUROC: 0.518

Summary


,Model,Accuracy,Macro AUROC
0,Default,24.00%,0.510
1,Typed Decisions,35.60%,0.518


## Fine-tuning Default Laya on Training Dataset (ModernBERT encoder frozen)

In [22]:
# Official Laya package + dependencies
!pip install -q git+https://github.com/NandhaKishorM/laya.git
!pip install -q transformers datasets accelerate huggingface_hub sentencepiece

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [23]:
import os
import json
import torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, QTYPES

MODEL_ID = "convaiinnovations/laya"

model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))

with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

# Reduce context length for T4
cfg["max_len"] = 512

def build_item(state, qdef, gold):

    keys = list(qdef["criteria"].keys())

    seq, markers = build_sequence(
        tok,
        state,
        {
            "t": qdef["type"],
            "ins": qdef["instructions"],
            "crit": qdef["criteria"]
        },
        cfg["max_len"],
        cfg["head_max_len"]
    )

    if len(markers) != len(keys):
        return None

    return {
        "ids": seq,
        "markers": markers,
        "qtype": QTYPES[qdef["type"]],
        "target": [1.0 if k == gold else 0.0 for k in keys],
        "label": keys.index(gold)
    }

def build_split(df):

    items=[]

    for _,row in df.iterrows():

        state=row["input"]
        answers=row["target"]["rule_outputs"]

        for qid,q in questions.items():

            item=build_item(state,q,answers[qid])

            if item is not None:
                items.append(item)

    return items

train_items=build_split(finetune_df)
val_items=build_split(validation_df)

torch.save(train_items,"/kaggle/working/train_items.pt")
torch.save(val_items,"/kaggle/working/val_items.pt")

print(f"Train items: {len(train_items)}")
print(f"Validation items: {len(val_items)}")

Fetching 38 files:   0%|          | 0/38 [00:00<?, ?it/s]

Train items: 1000
Validation items: 250


In [24]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 512
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    
    
    
    
    model.encoder.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False})
    
    model.head_checkpointing = True
    # Freeze ModernBERT encoder completely
    for p in model.encoder.parameters():
        p.requires_grad = False
    
    model.to(device)
    model.train()
    ddp_model = DDP(
        model,
        device_ids=[local_rank],
        find_unused_parameters=False
    )






    
    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)

    # Hold the calibration slice out of training before sharding. Temperatures fitted on
    # items the run has already trained on measure the fit rather than the calibration: the
    # model is near-certain and near-correct on them, so the optimiser has nothing to soften
    # and returns a degenerate scale. The seed is fixed and rank-independent, so every rank
    # withholds exactly the same items and none of them reaches a training batch.
    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)
    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]
    my_items = train_items[rank::world_size]
    
    EPOCHS = 4
    MICRO_BATCH = 2      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 2      # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # GRPO baseline samples
    LR_ENCODER = 0.0  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    
    
    
    
    enc_params = [
    p for n, p in ddp_model.named_parameters()
    if "encoder." in n]
    
    head_params = [
    p for n, p in ddp_model.named_parameters()
    if "encoder." not in n
    ]
    
    optimizer = torch.optim.AdamW(
        head_params,
        lr=LR_HEAD,
        weight_decay=0.01,
        foreach=False
    )



    
    
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            # 1. Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            # 2. Evaluate proper scoring reward (w_sph=0.75 for soft target matching)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            # 3. Policy gradient loss + full 1.0 soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()
    
    # Post-training temperature calibration on rank 0 (micro-batched in chunks of 16 to prevent OOM)
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        # calib_items was held out above and never entered a training batch
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))
        
        fitted_temps = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-typed-decisions"
        cfg["temperature"] = fitted_temps
        # This fit is per type; inherited bucket overrides would hide the new values.
        cfg.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

Writing /kaggle/working/train_ddp.py


In [25]:
## Finetuning Default Laya


import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


OUTPUT_DIR = "/kaggle/working/laya_rule_engine"

!torchrun \
    --standalone \
    --nproc_per_node=2 \
    /kaggle/working/train_ddp.py \
    "{model_dir}" \
    "{OUTPUT_DIR}"

W0927 13:19:00.717000 211 torch/distributed/run.py:852] 
W0927 13:19:00.717000 211 torch/distributed/run.py:852] *****************************************
W0927 13:19:00.717000 211 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0927 13:19:00.717000 211 torch/distributed/run.py:852] *****************************************
[W927 13:19:00.985929711 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W927 13:19:08.394376906 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W927 13:19:08.582933439 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Starting 2xT4 DDP training: 900 train items (100 held out for calibration) | 450 per rank | 4 epochs
[rank0]:W0927 13:19:29.538000 219 torch/_inductor/

## Evaluating Finetuned Default Laya

In [26]:
import laya

# Load the fine-tuned checkpoint
ft_model = laya.load("/kaggle/working/laya_rule_engine")

print("Fine-tuned model loaded.")

/usr/local/lib/python3.12/dist-packages/laya/agent.py:928: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using temperature[0]=10.0 -> 5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


Fine-tuned model loaded.


In [27]:
import pandas as pd
from tqdm.auto import tqdm

results = []

for _, row in tqdm(validation_df.iterrows(), total=len(validation_df), desc="Fine-tuned"):

    state = {
        "document_context": row["input"]["document_context"],
        "email_thread": row["input"]["email_thread"],
        "document_text": row["input"]["document_text"],
        "reference_records": row["input"]["reference_records"],
        "decision_policy": row["input"]["decision_policy"]
    }

    pred = ft_model.system_one(
        state,
        questions,
        max_len=512
    )

    answers = pred["answers"]
    gt = row["target"]["rule_outputs"]

    for rule in questions.keys():
        results.append({
            "case_id": row["case_id"],
            "rule": rule,
            "ground_truth": gt[rule],
            "prediction": answers[rule]["choice"],
            "confidence": answers[rule]["confidence"],
            "correct": answers[rule]["choice"] == gt[rule]
        })

finetuned_results = pd.DataFrame(results)

finetuned_results.head()

Fine-tuned:   0%|          | 0/50 [00:00<?, ?it/s]

,case_id,rule,ground_truth,prediction,confidence,correct
0,case_ab6e3f1b9c2e7640,R01_AMENDMENT,DO_NOT_APPLY,APPLY,0.0001,False
1,case_ab6e3f1b9c2e7640,R02_SUBSTITUTION,REJECT,ACCEPT,0.0001,False
2,case_ab6e3f1b9c2e7640,R03_RELEASE,NOT_APPLICABLE,RELEASE,0.0003,False
3,case_ab6e3f1b9c2e7640,R04_DOCUMENT_RELATION,CORRECTION,CORRECTION,0.0005,True
4,case_ab6e3f1b9c2e7640,R05_GOVERNING_TERMS,REVIEW,ORIGINAL,0.0003,False


In [28]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

print("="*70)
print("Fine-tuned Laya")
print("="*70)

# Overall accuracy
overall_acc = finetuned_results["correct"].mean()
print(f"Overall Accuracy: {overall_acc:.2%}")

# Per-rule accuracy
print("\nPer-rule Accuracy")
rule_acc = finetuned_results.groupby("rule")["correct"].mean().sort_index()
print((rule_acc * 100).round(2).astype(str) + "%")

# Per-rule AUROC
print("\nPer-rule AUROC")

aucs = {}

for rule, g in finetuned_results.groupby("rule"):
    y_true = g["correct"].astype(int)

    if len(np.unique(y_true)) < 2:
        aucs[rule] = np.nan
    else:
        aucs[rule] = roc_auc_score(y_true, g["confidence"])

auc_series = pd.Series(aucs).sort_index()

print(auc_series.round(3))
print(f"\nMacro AUROC: {auc_series.mean():.3f}")

Fine-tuned Laya
Overall Accuracy: 24.00%

Per-rule Accuracy
rule
R01_AMENDMENT            26.0%
R02_SUBSTITUTION         30.0%
R03_RELEASE              18.0%
R04_DOCUMENT_RELATION    24.0%
R05_GOVERNING_TERMS      22.0%
Name: correct, dtype: object

Per-rule AUROC
R01_AMENDMENT            0.561
R02_SUBSTITUTION         0.415
R03_RELEASE              0.352
R04_DOCUMENT_RELATION    0.602
R05_GOVERNING_TERMS      0.543
dtype: float64

Macro AUROC: 0.495


In [29]:
comparison = pd.DataFrame({
    "Model": [
        "Zero-shot Laya",
        "Fine-tuned Laya"
    ],
    "Accuracy": [
        default_metrics["accuracy"],
        overall_acc
    ],
    "Macro AUROC": [
        default_metrics["macro_auroc"],
        auc_series.mean()
    ]
})

display(comparison.style.format({
    "Accuracy": "{:.2%}",
    "Macro AUROC": "{:.3f}"
}))

,Model,Accuracy,Macro AUROC
0,Zero-shot Laya,24.00%,0.510
1,Fine-tuned Laya,24.00%,0.495
